# 07 · Independent reproduction of the selected Phase 1 run (for Rahaf)
Re-trains the selected configuration once on another machine/account with the committed config and seed, evaluates it,
writes `results/phase1/reproduction/<run>/metrics.json` and compares it with the committed `results/phase1/<run>/metrics.json`
(delta in percentage points). Nothing here changes the committed results.

## 0. Environment (Colab)
Clone the repository, install pinned requirements, set `FETALVISION_DATA`, mount Google Drive and unzip the Zenodo
archive (`FETAL_PLANES_ZENODO.zip`, record 3904280, CC BY 4.0) into `$FETALVISION_DATA/fetal_planes/raw/`.
All logic lives in the `fetalvision` package; this notebook only calls its CLIs.

In [ ]:
import os, subprocess, pathlib
if not pathlib.Path('FetalVision').exists():
    !git clone https://github.com/lamooooo1418/FetalVision.git
%cd FetalVision
!pip -q install -r requirements.txt -e .
os.environ['FETALVISION_DATA'] = '/content/data'
pathlib.Path('/content/data/fetal_planes/raw').mkdir(parents=True, exist_ok=True)
print('FETALVISION_DATA =', os.environ['FETALVISION_DATA'])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/fetal planes/FETAL_PLANES_ZENODO.zip'   # adjust to where the Zenodo zip lives on your Drive
!test -f "$ZIP" && unzip -q -n "$ZIP" -d "$FETALVISION_DATA/fetal_planes/raw/extracted" || echo "zip not found: download https://zenodo.org/records/3904280"
!ls "$FETALVISION_DATA/fetal_planes/raw/extracted" | head

In [ ]:
# Weights & Biases: paste your API key when prompted (or leave WANDB_MODE=offline and `wandb sync` later)
import os
try:
    import wandb; wandb.login()
except Exception as e:
    os.environ['WANDB_MODE'] = 'offline'; print('W&B offline:', e)

## 1. Which run? (read from the committed model_selection.json; override `RUN` if needed)

In [ ]:
import json, yaml, pathlib
sel = json.load(open('results/phase1/model_selection.json')); RUN = sel['selected_run']; print('selected run:', RUN)
CFG = next(p for p in pathlib.Path('configs').glob('phase1_*.yaml') if yaml.safe_load(open(p)).get('run_name') == RUN); print('config:', CFG)
cfg = yaml.safe_load(open(CFG)); PROCESSED = pathlib.Path(cfg['data']['processed_dir'].replace('${FETALVISION_DATA}', '')).name; print('processed split:', PROCESSED)

## 2. Data (same pipeline, same seed)

In [ ]:
STRATEGY = PROCESSED.replace('_nodedup', ''); ND = 'off' if PROCESSED.endswith('_nodedup') else 'on'
!python -m fetalvision.data.prepare_planes --config configs/phase1_data.yaml --split-strategy {STRATEGY} --near-dup {ND}

## 3. Train + evaluate once (results go to results/phase1/reproduction/, models to models/reproduction/)

In [ ]:
!python -m fetalvision.classification.train --config {CFG} --results-root results/reproduction --models-dir models/reproduction
!mkdir -p results/phase1/reproduction && cp -r results/reproduction/phase1/{RUN} results/phase1/reproduction/ && ls results/phase1/reproduction/{RUN}

## 4. Comparison with the committed metrics (percentage points)

In [ ]:
import json, pandas as pd
ref = json.load(open(f'results/phase1/{RUN}/metrics.json')); rep = json.load(open(f'results/phase1/reproduction/{RUN}/metrics.json'))
rows = []
for split, a, b in [('val', ref['val_metrics'], rep['val_metrics']), ('test', ref['metrics'], rep['metrics'])]:
    for k in ['accuracy', 'macro_f1', 'top2_accuracy', 'ece_10bins']:
        rows.append({'split': split, 'metric': k, 'committed': a[k], 'reproduction': b[k], 'delta_pp': 100 * (b[k] - a[k])})
    rows.append({'split': split, 'metric': 'log_loss', 'committed': a['log_loss'], 'reproduction': b['log_loss'], 'delta_pp': None})
    for c in ['Abdomen', 'Brain', 'Femur', 'Thorax']:
        rows.append({'split': split, 'metric': f'recall_{c}', 'committed': a['per_class'][c]['recall'], 'reproduction': b['per_class'][c]['recall'],
                     'delta_pp': 100 * (b['per_class'][c]['recall'] - a['per_class'][c]['recall'])})
df = pd.DataFrame(rows); print('hardware committed:', ref['hardware'].get('device'), ref['hardware'].get('cpu'), '| reproduction:', rep['hardware'].get('device'))
print('committed test CI95:', ref['metrics']['accuracy_ci95_wilson'], '| reproduction inside CI:', ref['metrics']['accuracy_ci95_wilson'][0] <= rep['metrics']['accuracy'] <= ref['metrics']['accuracy_ci95_wilson'][1])
df.to_csv(f'results/phase1/reproduction/{RUN}/comparison.csv', index=False); df